<a href="https://colab.research.google.com/github/RamonMedina04/Sistemas-Inteligentes/blob/main/Tarea_Redes_Neuronales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Importar bibliotecas

Usamos las mismas bibliotecas del notebook Titanic: pandas, TensorFlow, separación de datos y preprocesamiento.

In [4]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Leer y explorar los archivos

train contiene la respuesta Transported; test no la contiene. Exploramos el número de observaciones, algunas filas y los valores faltantes.

In [5]:
train = pd.read_csv("/content/train (1).csv")
test = pd.read_csv("/content/test (1).csv")

print(train.shape)
print(train.head())
print(train.isna().sum())
print(train["Transported"].value_counts())

(8693, 14)
  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4       

## 3. Seleccionar las entradas y la salida

Separamos la variable que queremos predecir (y) de las características (X). Quitamos identificador, nombre y cabina, igual que la práctica Titanic hace con columnas difíciles de utilizar directamente. Separamos 80% para entrenar y 20% para validar.

In [6]:
X = train.drop(columns=["Transported", "PassengerId", "Name", "Cabin"])
y = train["Transported"].astype(int)

X_final = test.drop(columns=["PassengerId", "Name", "Cabin"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_final = X_final.reset_index(drop=True)

## 4. Rellenar valores faltantes

Para variables categóricas usamos la moda y para variables numéricas usamos la media, tal como se hace en Titanic. Calculamos ambas solamente con el conjunto de entrenamiento para evitar fuga de información.

In [7]:
categoricas = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
numericas = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

for columna in categoricas:
    moda = X_train[columna].mode().iloc[0]
    X_train[columna] = X_train[columna].fillna(moda)
    X_val[columna] = X_val[columna].fillna(moda)
    X_final[columna] = X_final[columna].fillna(moda)

for columna in numericas:
    promedio = X_train[columna].mean()
    X_train[columna] = X_train[columna].fillna(promedio)
    X_val[columna] = X_val[columna].fillna(promedio)
    X_final[columna] = X_final[columna].fillna(promedio)

/tmp/ipykernel_1991/2994281380.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train[columna] = X_train[columna].fillna(moda)
/tmp/ipykernel_1991/2994281380.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_val[columna] = X_val[columna].fillna(moda)
/tmp/ipykernel_1991/2994281380.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

## 5. Convertir categorías a números

OneHotEncoder representa cada categoría con columnas 0/1, sin imponer un orden numérico artificial entre los planetas, destinos o valores booleanos. Los codificadores se ajustan solo con X_train.

In [8]:
for columna in categoricas:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoder.fit(X_train[[columna]])

    nombres = [columna + "_" + str(v) for v in encoder.categories_[0]]

    X_train = pd.concat([
        X_train.drop(columns=columna),
        pd.DataFrame(encoder.transform(X_train[[columna]]), columns=nombres)
    ], axis=1)

    X_val = pd.concat([
        X_val.drop(columns=columna),
        pd.DataFrame(encoder.transform(X_val[[columna]]), columns=nombres)
    ], axis=1)

    X_final = pd.concat([
        X_final.drop(columns=columna),
        pd.DataFrame(encoder.transform(X_final[[columna]]), columns=nombres)
    ], axis=1)

## 6. Estandarizar las variables numéricas

StandardScaler transforma variables numéricas a una escala comparable. Los datos quedan numéricos y listos para TensorFlow.

In [9]:
scaler = StandardScaler()
scaler.fit(X_train[numericas])

X_train[numericas] = scaler.transform(X_train[numericas])
X_val[numericas] = scaler.transform(X_val[numericas])
X_final[numericas] = scaler.transform(X_final[numericas])

X_train = X_train.astype("float32")
X_val = X_val.astype("float32")
X_final = X_final.astype("float32")

print("Número de entradas:", X_train.shape[1])

Número de entradas: 16


## 7. Crear la red neuronal

La capa oculta con 16 neuronas ReLU aprende relaciones no lineales. La salida con una neurona sigmoide estima la probabilidad de Transported=True.

In [10]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 289 (1.13 KB)

 Trainable params: 289 (1.13 KB)

 Non-trainable params: 0 (0.00 B)

## 8. Configurar y entrenar

Adam actualiza pesos y sesgos. Binary crossentropy mide el error de una clasificación binaria. Accuracy permite observar la proporción de aciertos. Validation_data usa ejemplos que no actualizan los pesos.

In [11]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val)
)

Epoch 1/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7088 - loss: 0.5936 - val_accuracy: 0.7522 - val_loss: 0.5284
Epoch 2/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7751 - loss: 0.4936 - val_accuracy: 0.7838 - val_loss: 0.4707
Epoch 3/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7882 - loss: 0.4596 - val_accuracy: 0.7913 - val_loss: 0.4499
Epoch 4/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7883 - loss: 0.4472 - val_accuracy: 0.7941 - val_loss: 0.4424
Epoch 5/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7900 - loss: 0.4426 - val_accuracy: 0.7941 - val_loss: 0.4384
Epoch 6/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7899 - loss: 0.4396 - val_accuracy: 0.7982 - val_loss: 0.4375
Epoch 7/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7919 - loss: 0.4375 - val_accuracy: 0.7878 - val_loss: 0.4377
Epoch 8/30
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7931 - loss: 0.4364 - val_accuracy: 0.

## 9. Evaluar con el conjunto de validación

El modelo devuelve probabilidades. Las convertimos en clases con un umbral de 0.5 y calculamos una matriz de confusión, igual que en Titanic.

In [12]:
probabilidades = model.predict(X_val)
predicciones = (probabilidades > 0.5).astype(int).ravel()

matriz = tf.math.confusion_matrix(labels=y_val, predictions=predicciones)
print("Matriz de confusión:")
print(matriz.numpy())

55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Matriz de confusión:
[[674 189]
 [164 712]]


## 10. Predecir el archivo test y guardar CSV

Aplicamos al test las mismas transformaciones aprendidas en el entrenamiento y generamos el archivo de entrega con las columnas PassengerId y Transported.

In [13]:
probabilidades_test = model.predict(X_final)
transportados = (probabilidades_test > 0.5).ravel()

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": transportados
})

submission.to_csv("submission_sencilla.csv", index=False)
print(submission.head())

134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
